In [15]:
import os
import re
import uuid

import jwt
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

# ====== CONFIG ======
BASE_URL = "http://localhost:7001"

TENANT_ID     = os.getenv("AZ_TENANT_ID",     "<tenant-id>")
CLIENT_ID     = os.getenv("AZ_CLIENT_ID",     "<client-id>")
OAUTH_SECRET = os.getenv("AZ_CLIENT_SECRET", "<client-secret>")
OAUTH_SCOPE   = os.getenv("AZ_OAUTH_SCOPE",   "3db474b9-6a0c-4840-96ac-1fceb342124f/.default")
ISSUER_API_URL = os.getenv("ISSUER_API_URL", "https://<host>/<path>")
DID_AUTHORITY = os.getenv("DID_AUTHORITY", "<did-authority>")
MANIFEST = os.getenv("MANIFEST", "<manifest>")
CALLBACK_URL = os.getenv("CALLBACK_URL", "<callback-url>")
# ===================


# check if container is up and running
try:
    response = requests.get(BASE_URL, timeout=5)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Connection error: {e}")

In [16]:
''' Create the verification request URI '''

def _normalize_scope(scope: str) -> str:
    s = scope.strip()
    # add /.default if missing
    if not s.endswith("/.default"):
        s = s.rstrip("/") + "/.default"
    return s

def get_oauth_token(tenant_id, client_id, client_secret, scope):
    """
    OAuth2 client_credentials verso il token endpoint v2 of Microsoft
    """
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    data = {
        "client_id": client_id,
        "client_secret": client_secret,
        "grant_type": "client_credentials",
        "scope": _normalize_scope(scope),
    }
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    resp = requests.post(url, data=data, headers=headers, timeout=20)
    try:
        resp.raise_for_status()
    except requests.HTTPError:
        raise SystemExit(f"[TOKEN ERROR] {resp.status_code} {resp.text}")
    token = resp.json().get("access_token")

    if not token:
        raise SystemExit("[TOKEN ERROR] access_token missing in the response!")
    return token

payload = {
    "authority": DID_AUTHORITY,
    "callback": {
        "url": CALLBACK_URL,
        "state": str(uuid.uuid4()),
        "headers": {
          "api-key": "callback-api-key-SPIKEREPLY2025"
        }
    },
    "registration": {
        "clientName": "MS Entra Verified ID"
    },
    "includeReceipt": False,
    "requestedCredentials": [
        {
            "type": "AdminCredential",
            "purpose": "To verify that you have admin authorization",
            "acceptedIssuers": [DID_AUTHORITY],
            "configuration": {
                "validation": {
                    "allowRevoked": False,
                    "validateLinkedDomain": True
                }
            },
            "constraints": [
                {
                    "claimName": "admin",
                    "values": ["true"]
                }
            ]
        }
    ]
}

presentation_request = requests.post(
    url="https://verifiedid.did.msidentity.com/v1.0/verifiableCredentials/createPresentationRequest",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {get_oauth_token(TENANT_ID, CLIENT_ID, OAUTH_SECRET, OAUTH_SCOPE)}"
    },
    json=payload,
    timeout=15
).json()

try:
    req_uri = re.search(r'request_uri=([^ ]+)', presentation_request["url"]).group(1)
except AttributeError:
    req_uri = None
    exit(-1)

# IN QUESTO REQ_URI TROVO LA REQUEST ID
req_uri
# QUESTO REQ_URI VA GIRATO AD ALICE

'https://verifiedid.did.msidentity.com/v1.0/tenants/070bd82a-c7ba-4bed-bc50-3dca476dc820/verifiableCredentials/presentationRequests/3245c82e-f143-4284-a98e-80df6532432f'

In [17]:
# ============================= ALICE PART =============================
# TODO: FILL WITH WALTID AGENT DATA
'''Create and login to walt.id account, then get the wallet ID'''

body = {
  "type": "email",
  "name": "Alice",
  "email": "alice@walt.id",
  "password": "aaa"
}

resp = requests.post(
    BASE_URL + "/wallet-api/auth/register",
    headers={"Content-Type": "application/json"},
    json=body,
    timeout=15
)


body = {
  "type": "email",
  "email": "alice@walt.id",
  "password": "aaa"
}

resp = requests.post(
    BASE_URL + "/wallet-api/auth/login",
    headers={"Content-Type": "application/json"},
    json=body,
    timeout=15
)

# if authN is successful, then save the token
waltid_token = resp.json().get("token") if resp.status_code == 200 else None


resp = requests.get(
    BASE_URL + "/wallet-api/wallet/accounts/wallets",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {waltid_token}"
    },
    timeout=15
)

# if code = 200 get the wallet id
wallet_id = resp.json().get("wallets")[0].get("id") if resp.status_code == 200 else None

wallet_id   # '9c9dea06-969a-4a13-86a1-9b10630a00a8'

'9c9dea06-969a-4a13-86a1-9b10630a00a8'

In [18]:
# here it arrives "requestStatus": "request_retrieved" on the CALLBACK side

req_obj_jwt = requests.get(url=req_uri, headers={"Accept": "application/jwt"}).text

req_obj = jwt.decode(req_obj_jwt, options={"verify_signature": False})
req_obj

{'jti': '03d93d8d-659e-4dda-bb02-dc619f877fe2',
 'iat': 1771254921,
 'response_type': 'id_token',
 'response_mode': 'post',
 'scope': 'openid',
 'nonce': 'LDLQjvf00cUJ3ulOqH7Mkw==',
 'redirect_uri': 'https://verifiedid.did.msidentity.com/v1.0/tenants/070bd82a-c7ba-4bed-bc50-3dca476dc820/verifiableCredentials/verifyPresentation',
 'registration': {'client_name': 'MS Entra Verified ID',
  'subject_syntax_types_supported': ['did:ion', 'did:jwk'],
  'vp_formats': {'jwt_vp': {'alg': ['ES256',
     'ES384',
     'ES512',
     'ES256K',
     'EdDSA']},
   'jwt_vc': {'alg': ['ES256', 'ES384', 'ES512', 'ES256K', 'EdDSA']}}},
 'client_id': 'did:web:stgacctestdid.z38.web.core.windows.net',
 'state': 'djPb1qfKNefMOjKMcB0tM/KBp4HQ4OAuWgrP5y5+c5IBIMLlxI3FqgwuV3Zl7mhwasHE1B8JOOPn2v6pPfdXgv1JpHosLJIUFVM8lR1d9JtgdDLE69ouVAIryZZEAt7Z/KRZXNIy6w2UtqGT1CJJFLeagBpE3kkAWcw6Ny/twm/p6l5uugynzZ9er21fvah5g0mZaJnCftABfwvkupr+1aT+9q1zaB4p7XW2cBFpNGJAcGTAswnvOftfrCnC+Z3hfeAOvFCNqhbFXihZyYIaJwemIaQBp2sV9iUkNzDjdZGts

In [19]:
# extract the credential that matches starting from the presentation definition.

data = req_obj["claims"]["vp_token"]["presentation_definition"]

pres_def = {**data, "input_descriptors": [{k: v for k, v in d.items() if k != "schema"} for d in data.get("input_descriptors", [])]}
print(pres_def)
for item in pres_def["input_descriptors"][0]["constraints"]["fields"]:
    if item["path"] == ['$.vc.credentialSubject.admin']:
        # substitute 'pattern': '/^true$/gi' with 'pattern': 'true' (respectively with 'false')
        m = re.search(r'^/\^(true|false)\$/gi$', item["filter"]["pattern"], flags=re.I)
        item["filter"]["pattern"] = m.group(1) if m else None

print(pres_def)

resp = requests.post(
    url=f"http://localhost:7001/wallet-api/wallet/{wallet_id}/exchange/matchCredentialsForPresentationDefinition",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {waltid_token}",},
    json=pres_def,
    timeout=15
)

print(resp.text)

credentialId = resp.json()[0]["id"] if resp.json() else []
print(credentialId)

{'id': '205f580d-f3bf-4c03-add0-10c314de7411', 'input_descriptors': [{'id': '324435e4-a942-4a12-a85a-c1bc6da4f2fc', 'name': 'AdminCredential', 'purpose': 'To verify that you have admin authorization', 'constraints': {'fields': [{'path': ['$.issuer', '$.vc.issuer', '$.iss'], 'filter': {'type': 'string', 'pattern': 'did:web:stgacctestdid.z38.web.core.windows.net'}}, {'path': ['$.vc.credentialSubject.admin'], 'filter': {'type': 'string', 'pattern': '/^true$/gi'}}]}}]}
{'id': '205f580d-f3bf-4c03-add0-10c314de7411', 'input_descriptors': [{'id': '324435e4-a942-4a12-a85a-c1bc6da4f2fc', 'name': 'AdminCredential', 'purpose': 'To verify that you have admin authorization', 'constraints': {'fields': [{'path': ['$.issuer', '$.vc.issuer', '$.iss'], 'filter': {'type': 'string', 'pattern': 'did:web:stgacctestdid.z38.web.core.windows.net'}}, {'path': ['$.vc.credentialSubject.admin'], 'filter': {'type': 'string', 'pattern': 'true'}}]}}]}
[{"wallet":"9c9dea06-969a-4a13-86a1-9b10630a00a8","id":"250c0f11-5

To see how to continue the verification process, look directly at the `__main__` code of Alice (`submit_verifiable_presentation`)
